In [12]:
# Imports and Setup
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Paths
SILVER_PATH = "../data/silver/"
GOLD_PATH = "../data/gold/"

if not os.path.exists(GOLD_PATH):
    os.makedirs(GOLD_PATH)

In [13]:
cert_silver = pd.read_csv(os.path.join(SILVER_PATH, "certifications_silver.csv"))
clubs_silver = pd.read_csv(os.path.join(SILVER_PATH, "clubs_silver.csv"))
results_silver = pd.read_csv(os.path.join(SILVER_PATH, "results_silver.csv"))

In [14]:
dim_athlete = cert_silver[['Code', 'person_type_clean', 'gender_clean', 'DOB', 'Age', 
                             'Mental Handicap (SOB has this certificate)', 
                             'Parents Consent (SOB has this certificate)',
                             'HAP (SOB has this certificate)',
                             'Unified Partner (SOB has this certificate)',
                             'dob_missing', 'person_type_missing']].copy()

dim_athlete = dim_athlete.rename(columns={
    'Code': 'athlete_id',
    'person_type_clean': 'person_type',
    'gender_clean': 'gender',
    'DOB': 'date_of_birth',
    'Mental Handicap (SOB has this certificate)': 'mental_handicap_flag',
    'Parents Consent (SOB has this certificate)': 'parents_consent_flag',
    'HAP (SOB has this certificate)': 'hap_flag',
    'Unified Partner (SOB has this certificate)': 'unified_partner_flag'
})

dim_athlete = dim_athlete.drop_duplicates(subset=['athlete_id'], keep='first')

dim_athlete['athlete_quality_flag'] = ((dim_athlete['dob_missing'] == 0) & 
                                        (dim_athlete['person_type_missing'] == 0)).astype(int)

dim_athlete['gold_created_timestamp'] = datetime.now()

dim_athlete.to_csv(os.path.join(GOLD_PATH, "dim_athlete.csv"), index=False)

In [15]:
dim_sport = results_silver[['sport_clean']].drop_duplicates().copy()
dim_sport = dim_sport.dropna()

dim_sport = dim_sport.reset_index(drop=True)
dim_sport['sport_id'] = range(1, len(dim_sport) + 1)

dim_sport = dim_sport.rename(columns={'sport_clean': 'sport_name'})
dim_sport = dim_sport[['sport_id', 'sport_name']]

dim_sport['gold_created_timestamp'] = datetime.now()

dim_sport.to_csv(os.path.join(GOLD_PATH, "dim_sport.csv"), index=False)

In [16]:
dim_region = clubs_silver[['region']].drop_duplicates().copy()
dim_region = dim_region.dropna()
dim_region = dim_region.reset_index(drop=True)

dim_region['region_id'] = range(1, len(dim_region) + 1)
dim_region = dim_region.rename(columns={'region': 'region_name'})
dim_region = dim_region[['region_id', 'region_name']]

dim_region['gold_created_timestamp'] = datetime.now()

dim_region.to_csv(os.path.join(GOLD_PATH, "dim_region.csv"), index=False)

In [17]:
dim_club = clubs_silver[['club_id', 'club_name', 'region', 'total_participations']].copy()

dim_club = dim_club.merge(dim_region, left_on='region', right_on='region_name', how='left')

athlete_per_club = cert_silver[cert_silver['person_type_missing'] == 0].groupby('Club')['Code'].count().reset_index()
athlete_per_club = athlete_per_club.rename(columns={'Club': 'club_name', 'Code': 'certified_athlete_count'})

dim_club = dim_club.merge(athlete_per_club, on='club_name', how='left')
dim_club['certified_athlete_count'] = dim_club['certified_athlete_count'].fillna(0).astype(int)

dim_club = dim_club[['club_id', 'club_name', 'region_id', 'region_name', 'total_participations', 'certified_athlete_count']]

dim_club['gold_created_timestamp'] = datetime.now()

dim_club.to_csv(os.path.join(GOLD_PATH, "dim_club.csv"), index=False)

In [18]:
fact_results = results_silver[['Code', 'year', 'sport_clean', 'Club', 
                                'rank_numeric', 'score_numeric', 'is_disqualified']].copy()

fact_results = fact_results.rename(columns={
    'Code': 'athlete_id',
    'year': 'competition_year',
    'sport_clean': 'sport_name',
    'Club': 'club_name',
    'rank_numeric': 'rank',
    'score_numeric': 'score'
})

fact_results = fact_results.merge(dim_sport[['sport_id', 'sport_name']], 
                                  on='sport_name', how='left')

fact_results = fact_results.merge(dim_club[['club_id', 'club_name']], 
                                  on='club_name', how='left')

fact_results['score_missing_flag'] = fact_results['score'].isna().astype(int)
fact_results['rank_missing_flag'] = fact_results['rank'].isna().astype(int)

fact_results = fact_results[['athlete_id', 'competition_year', 'sport_id', 'sport_name',
                              'club_id', 'club_name', 'rank', 'score', 'is_disqualified',
                              'score_missing_flag', 'rank_missing_flag']]

fact_results['gold_created_timestamp'] = datetime.now()

fact_results.to_csv(os.path.join(GOLD_PATH, "fact_athlete_results.csv"), index=False)

In [19]:
fact_participation = results_silver.groupby(['Code', 'year', 'Club']).agg({
    'sport_clean': 'count',  
    'rank_numeric': 'count'  
}).reset_index()

fact_participation = fact_participation.rename(columns={
    'Code': 'athlete_id',
    'year': 'participation_year',
    'Club': 'club_name',
    'sport_clean': 'competition_count',
    'rank_numeric': 'valid_rank_count'
})

sports_per_athlete_year = results_silver.groupby(['Code', 'year'])['sport_clean'].nunique().reset_index()
sports_per_athlete_year = sports_per_athlete_year.rename(columns={
    'Code': 'athlete_id',
    'year': 'participation_year',
    'sport_clean': 'unique_sport_count'
})

fact_participation = fact_participation.merge(sports_per_athlete_year, 
                                              on=['athlete_id', 'participation_year'], 
                                              how='left')

fact_participation['multi_sport_flag'] = (fact_participation['unique_sport_count'] > 1).astype(int)

fact_participation = fact_participation.merge(dim_club[['club_id', 'club_name']], 
                                              on='club_name', how='left')

fact_participation = fact_participation[['athlete_id', 'participation_year', 'club_id', 'club_name',
                                         'competition_count', 'valid_rank_count', 
                                         'unique_sport_count', 'multi_sport_flag']]

fact_participation['gold_created_timestamp'] = datetime.now()

fact_participation.to_csv(os.path.join(GOLD_PATH, "fact_athlete_participation.csv"), index=False)

In [20]:
athlete_career = results_silver.groupby('Code').agg({
    'year': ['min', 'max', 'nunique'],
    'rank_numeric': 'count',
    'score_numeric': ['count', 'mean', 'max', 'min'],
    'is_disqualified': 'sum',
    'sport_clean': 'nunique'
}).reset_index()

athlete_career.columns = ['athlete_id', 'first_year', 'last_year', 'years_active',
                          'total_competitions', 'competitions_with_score', 'avg_score',
                          'best_score', 'worst_score', 'disqualification_count', 'unique_sports']

athlete_career['career_span_years'] = athlete_career['last_year'] - athlete_career['first_year'] + 1

athlete_career = athlete_career.merge(dim_athlete[['athlete_id', 'person_type', 'gender', 'Age']], 
                                     on='athlete_id', how='left')

favorite_sport = results_silver.groupby(['Code', 'sport_clean']).size().reset_index(name='count')
favorite_sport = favorite_sport.loc[favorite_sport.groupby('Code')['count'].idxmax()]
favorite_sport = favorite_sport.rename(columns={'Code': 'athlete_id', 'sport_clean': 'favorite_sport'})

athlete_career = athlete_career.merge(favorite_sport[['athlete_id', 'favorite_sport']], 
                                     on='athlete_id', how='left')

athlete_career['gold_created_timestamp'] = datetime.now()

athlete_career.to_csv(os.path.join(GOLD_PATH, "athlete_career.csv"), index=False)

In [21]:
club_metrics = results_silver.groupby('Club').agg({
    'Code': 'nunique',
    'rank_numeric': 'count',
    'score_numeric': ['mean', 'max'],
    'is_disqualified': 'sum',
    'year': 'nunique'
}).reset_index()

club_metrics.columns = ['club_name', 'unique_athletes', 'total_competitions',
                        'avg_score', 'best_score', 'disqualification_count', 'years_participated']

club_metrics = club_metrics.merge(dim_club[['club_id', 'club_name', 'region_id', 'region_name',
                                            'total_participations', 'certified_athlete_count']], 
                                 on='club_name', how='left')

club_metrics = club_metrics[['club_id', 'club_name', 'region_id', 'region_name',
                             'unique_athletes', 'certified_athlete_count', 'total_competitions',
                             'years_participated', 'avg_score', 'best_score',
                             'disqualification_count', 'total_participations']]

club_metrics['gold_created_timestamp'] = datetime.now()

club_metrics.to_csv(os.path.join(GOLD_PATH, "club_metrics.csv"), index=False)

In [22]:
annual_perf = results_silver.groupby(['year', 'sport_clean']).agg({
    'Code': 'nunique',
    'rank_numeric': 'count',
    'score_numeric': ['count', 'mean', 'max', 'min'],
    'is_disqualified': 'sum',
    'Gender': lambda x: (x == 'Male').sum()
}).reset_index()

annual_perf.columns = ['year', 'sport_name', 'unique_athletes', 'total_results',
                       'results_with_score', 'avg_score', 'best_score', 'worst_score',
                       'disqualification_count', 'male_count']

annual_perf = annual_perf.merge(dim_sport[['sport_id', 'sport_name']], 
                               on='sport_name', how='left')

annual_perf = annual_perf[['sport_id', 'sport_name', 'year', 'unique_athletes', 'total_results',
                           'results_with_score', 'avg_score', 'best_score', 'worst_score',
                           'male_count', 'disqualification_count']]

annual_perf['gold_created_timestamp'] = datetime.now()

annual_perf.to_csv(os.path.join(GOLD_PATH, "annual_performance.csv"), index=False)